In [33]:
import os, subprocess
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()

MODEL = os.environ.get("MODEL_ID")

In [34]:
TOOLS = [
    {
        "name": "bash",
        "description": "Run a shell command",
        "input_schema": {
            "type": "object",
            "properties": {
                "command": {
                    "type": "string"
                }
            },
            "required": ["command"]
        }
    }
]

In [ ]:
query1 = "How many .md files are in the current directory?"

In [36]:
messages = [
    {
        "role": "user",
        "content": query1
    }
]

messages

[{'role': 'user',
  'content': 'How many .md files are in the current directory?'}]

In [37]:
response = client.messages.create(model = MODEL, max_tokens = 1000, tools = TOOLS, messages = messages)

### Tasks

#### 1. Find the tool_use block. Read its id and input["command"]

In [38]:
print(response.content[0].type)
print(response.content[0].id)
print(response.content[0].name)
print(response.content[0].input["command"])
print(response.content)

tool_use
toolu_01JPtWpvJhUu3jSn5DUUFpWf
bash
ls -1 *.md 2>/dev/null | wc -l; ls -1a | grep -i '\.md$'
[ToolUseBlock(id='toolu_01JPtWpvJhUu3jSn5DUUFpWf', caller=DirectCaller(type='direct'), input={'command': "ls -1 *.md 2>/dev/null | wc -l; ls -1a | grep -i '\\.md$'"}, name='bash', type='tool_use', toolset_name=None)]


#### 2. Run that command in a terminal and copy the output.

#### 3. Append the model's turn to messages with the assistant role, then append a user message containing one tool_result. It needs the matching tool_use_id and the output as content.

In [39]:
messages.append(
    {
        "role": "assistant",
        "content": response.content
    }
)

messages

[{'role': 'user',
  'content': 'How many .md files are in the current directory?'},
 {'role': 'assistant',
  'content': [ToolUseBlock(id='toolu_01JPtWpvJhUu3jSn5DUUFpWf', caller=DirectCaller(type='direct'), input={'command': "ls -1 *.md 2>/dev/null | wc -l; ls -1a | grep -i '\\.md$'"}, name='bash', type='tool_use', toolset_name=None)]}]

In [40]:
command = response.content[0].input["command"]
command

"ls -1 *.md 2>/dev/null | wc -l; ls -1a | grep -i '\\.md$'"

In [41]:
result = subprocess.run(command, shell=True, cwd=os.getcwd(),
                           capture_output=True, text=True, timeout=120).stdout
result

'       1\ns01_agent_loop.md\n'

In [42]:
messages.append(
    {
        "role": "user",
        "content": [
            {
                "type": "tool_result",
                "tool_use_id": response.content[0].id,
                "content": result
            }
        ]
    }
)

messages

[{'role': 'user',
  'content': 'How many .md files are in the current directory?'},
 {'role': 'assistant',
  'content': [ToolUseBlock(id='toolu_01JPtWpvJhUu3jSn5DUUFpWf', caller=DirectCaller(type='direct'), input={'command': "ls -1 *.md 2>/dev/null | wc -l; ls -1a | grep -i '\\.md$'"}, name='bash', type='tool_use', toolset_name=None)]},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'toolu_01JPtWpvJhUu3jSn5DUUFpWf',
    'content': '       1\ns01_agent_loop.md\n'}]}]

#### 4. Call client.messages.create(...) again and look at the new stop_reason.

In [43]:
response = client.messages.create(model = MODEL, max_tokens = 1000, tools = TOOLS, messages = messages)

In [44]:
response

Message(id='msg_011ChQXnYaSWJ3gLZzT2fdTU', container=None, content=[ThinkingBlock(signature='CAQS8gUKEAgUGAI4AUIIdGhpbmtpbmcSDOJk/nLajpYpgIXAmxoM+MXkf/+qHUL4NGkzIjBAAz1E7Ny/rXptqDv5HdvEzHj/q8EZDA+i8wrDwoTYxEJbbMAblcSw6iGEifFLMEEqjwUtZbfU+OdtKEmF97ZXU0iDNNIz88NpGr0QevLGVUmkcRujvVrfyTCjpuzDq9kcM3jSiASA5LeC0DHznZmYF2R/09OhLxcqT/F0mQ3s9IWaxNOxPGNj9n2G5yil09uTlUndVYG7rg6sWKSkhqEmyOr18B7p+Vg++MyhfDIlhQ0DmY1wKNZSL/TrJzJqg1SiHpc04Hm0ZXAtGVkXYnKITWAAofcRUVJ0Dno2hAkEXqTzvDUU5aTqoY9HMObwnRJ2fxnfqhDHvQ4UdsODTi2s4/ncRS8AxihKNtAHJgh6+0nBMMRBaH0HHsUz+WeZByQOuRCp3wUgbO3l98k/Cb1IHrb+qCO/Br4pNCErSp9IbhPxfrztTWtpwiRms2wqvy8uS/QkwOVkNXMwjSS0MEfdO+PN5Nfw1TA3MLjATgVm36XGQxb0JW9bE6MDgAf5Iox24Y0C3hLZZmmRSvmSLfVbaVWwRxaZZA2lNF8sohKGxYDY/SbjrNJwHPOnSrehf45k7J1ZLGpGnjkROQQ8zcOdbGtS4dr+YvH6z0kaDvOo7GgfPJAkt980n3kFjS8HSBU08RO1Oj0bHvUE9iJ6E0OnSm3pT0KIYeZlh0eIkEMnwqyMD+jtc3Ba8sS0cZlW88EOoOC5EIoEyt4UAmoOPogsiaBKrScU4MvgbSuzGA4ItnUAJxwI/9oQJKhFKzktOouj2xnLm+k8la/nYzi06GNxJelC4dZ6FmYn0XEf4Kce1eQiB2u3JLDBhtJqCZ7dz3l0XoR

In [45]:
print(response.content[0])
print(response.content[1])

ThinkingBlock(signature='CAQS8gUKEAgUGAI4AUIIdGhpbmtpbmcSDOJk/nLajpYpgIXAmxoM+MXkf/+qHUL4NGkzIjBAAz1E7Ny/rXptqDv5HdvEzHj/q8EZDA+i8wrDwoTYxEJbbMAblcSw6iGEifFLMEEqjwUtZbfU+OdtKEmF97ZXU0iDNNIz88NpGr0QevLGVUmkcRujvVrfyTCjpuzDq9kcM3jSiASA5LeC0DHznZmYF2R/09OhLxcqT/F0mQ3s9IWaxNOxPGNj9n2G5yil09uTlUndVYG7rg6sWKSkhqEmyOr18B7p+Vg++MyhfDIlhQ0DmY1wKNZSL/TrJzJqg1SiHpc04Hm0ZXAtGVkXYnKITWAAofcRUVJ0Dno2hAkEXqTzvDUU5aTqoY9HMObwnRJ2fxnfqhDHvQ4UdsODTi2s4/ncRS8AxihKNtAHJgh6+0nBMMRBaH0HHsUz+WeZByQOuRCp3wUgbO3l98k/Cb1IHrb+qCO/Br4pNCErSp9IbhPxfrztTWtpwiRms2wqvy8uS/QkwOVkNXMwjSS0MEfdO+PN5Nfw1TA3MLjATgVm36XGQxb0JW9bE6MDgAf5Iox24Y0C3hLZZmmRSvmSLfVbaVWwRxaZZA2lNF8sohKGxYDY/SbjrNJwHPOnSrehf45k7J1ZLGpGnjkROQQ8zcOdbGtS4dr+YvH6z0kaDvOo7GgfPJAkt980n3kFjS8HSBU08RO1Oj0bHvUE9iJ6E0OnSm3pT0KIYeZlh0eIkEMnwqyMD+jtc3Ba8sS0cZlW88EOoOC5EIoEyt4UAmoOPogsiaBKrScU4MvgbSuzGA4ItnUAJxwI/9oQJKhFKzktOouj2xnLm+k8la/nYzi06GNxJelC4dZ6FmYn0XEf4Kce1eQiB2u3JLDBhtJqCZ7dz3l0XoR4ZGoAZXhvnh9Gxm03nNdUy85CM5yYZYKqxchilPAysc+NDj/xuwx9HNam5cV3EgWNVFJ

In [46]:
print(response.stop_reason)
print(response.content[1].text)


end_turn
The current directory has **1** `.md` file: `s01_agent_loop.md`. That count includes hidden files but not subdirectories.


In [47]:
query2 = "Whats the capital of France?"

In [48]:
messages = [
    {
        "role": "user",
        "content": query2
    }
]

messages

[{'role': 'user', 'content': 'Whats the capital of France?'}]

In [49]:
response = client.messages.create(model = MODEL, max_tokens = 1000, tools = TOOLS, messages = messages)

In [50]:
print(response.content[0].type)
print(response.content)

text
[TextBlock(citations=None, text='The capital of France is Paris.', type='text')]


In [51]:
print(response.stop_reason)
print(response.content[0].text)

end_turn
The capital of France is Paris.
